In [1]:
import os
import html
import joblib
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

os.makedirs("models", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

In [2]:
df_raw = pd.read_csv("data/raw/SMSSpamCollection.csv", sep="\t", header=None, names=["label", "message"])

In [3]:
y = df_raw["label"].map({"ham": 0, "spam": 1})

# train test split. 80% training, 20% testing
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    df_raw, y, train_size=0.80, random_state=42, stratify=y
)
print(f"Train: {len(X_train_raw)}  Test: {len(X_test_raw)}")

Train: 4457  Test: 1115


In [4]:
vectorizer = CountVectorizer(stop_words="english", max_features=500)
X_train_words_raw = pd.DataFrame(
    vectorizer.fit_transform(X_train_raw["message"]).toarray(),
    columns=vectorizer.get_feature_names_out(), index=X_train_raw.index
)
X_test_words_raw = pd.DataFrame(
    vectorizer.transform(X_test_raw["message"]).toarray(),
    columns=vectorizer.get_feature_names_out(), index=X_test_raw.index
)

In [5]:
models = {
    "Majority-class baseline": DummyClassifier(strategy="most_frequent"),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": MultinomialNB(),
    "Random Forest (bagging)": RandomForestClassifier(random_state=42),
    "AdaBoost (boosting)": AdaBoostClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}

In [ ]:
mlflow.set_experiment("spam_filter_benchmarks_raw")
mlflow.sklearn.autolog()

results_autolog = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_words_raw, y_train_raw)
        predictions = model.predict(X_test_words_raw)
        test_metrics = {
            "Accuracy": accuracy_score(y_test_raw, predictions),
            "Precision": precision_score(y_test_raw, predictions, zero_division=0),
            "Recall": recall_score(y_test_raw, predictions, zero_division=0),
            "F1": f1_score(y_test_raw, predictions, zero_division=0),
        }
        mlflow.log_metrics({f"test_{k.lower()}": v for k, v in test_metrics.items()})
        results_autolog.append({"Model": name, **test_metrics})

mlflow.sklearn.autolog(disable=True)

In [7]:
results_df = pd.DataFrame(results_autolog).round(3)
print(results_df.to_string(index=False))

                  Model  Accuracy  Precision  Recall    F1
Majority-class baseline     0.866      0.000   0.000 0.000
          Decision Tree     0.957      0.863   0.805 0.833
                    KNN     0.930      1.000   0.477 0.645
            Naive Bayes     0.978      0.931   0.906 0.918
Random Forest (bagging)     0.974      0.948   0.852 0.898
    AdaBoost (boosting)     0.904      0.977   0.289 0.446
    Logistic Regression     0.977      0.984   0.839 0.906


In [8]:
df = pd.read_csv("data/raw/SMSSpamCollection.csv", sep="\t", header=None, names=["label", "message"])

In [9]:
# fixing the HTML bug found in the EDA (&lt;#&gt)
df["message"] = df["message"].apply(html.unescape)

In [10]:
df["char_len"] = df["message"].str.len()
df["word_len"] = df["message"].str.split().str.len()
df["digit_count"] = df["message"].str.count(r"\d")
df["exclaim_count"] = df["message"].str.count("!")
df["currency_count"] = df["message"].str.count(r"[£$€]")
df["capital_ratio"] = df["message"].apply(
    lambda x: sum(i.isupper() for i in x) / max(sum(i.isalpha() for i in x), 1)
)
engineered_cols = ["char_len", "word_len", "digit_count", "exclaim_count", "currency_count", "capital_ratio"]

In [11]:
df

,label,message,char_len,word_len,digit_count,exclaim_count,currency_count,capital_ratio
0,ham,"Go until jurong point, crazy.. Available only ...",111,20,0,0,0,0.036145
1,ham,Ok lar... Joking wif u oni...,29,6,0,0,0,0.111111
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,28,25,0,0,0.103093
3,ham,U dun say so early hor... U c already then say...,49,11,0,0,0,0.060606
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,13,0,0,0,0.042553
...,...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,160,30,21,1,1,0.089109
5568,ham,Will ü b going to esplanade fr home?,36,8,0,0,0,0.035714
5569,ham,"Pity, * was in mood for that. So...any other s...",57,10,0,0,0,0.048780
5570,ham,The guy did some bitching but I acted like i'd...,125,26,0,0,0,0.020202


In [12]:
# removing duplicate messages before splitting, so nothing appears in both train and test
before = len(df)
df = df.drop_duplicates(subset=["message", "label"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicates -> {len(df)} rows")

Dropped 403 duplicates -> 5169 rows


In [13]:
y = df["label"].map({"ham": 0, "spam": 1})

# train test split. 80% training, 20% testing
X_train_engraw, X_test_engraw, y_train, y_test = train_test_split(
    df, y, train_size=0.80, random_state=42, stratify=y
)
print(f"Train: {len(X_train_engraw)}  Test: {len(X_test_engraw)}")

Train: 4135  Test: 1034


In [14]:
X_train_engraw

,label,message,char_len,word_len,digit_count,exclaim_count,currency_count,capital_ratio
331,ham,"Ta-Daaaaa! I am home babe, are you still up ?",45,10,0,1,0,0.093750
383,ham,Yup having my lunch buffet now.. U eat already?,47,9,0,0,0,0.055556
573,ham,Ok anyway no need to change with what you said,46,10,0,0,0,0.027027
1346,ham,All e best 4 ur exam later.,27,7,1,0,0,0.052632
1070,ham,Now only i reached home. . . I am very tired n...,71,17,0,0,0,0.060000
...,...,...,...,...,...,...,...,...
598,spam,XCLUSIVE@CLUBSAISAI 2MOROW 28/5 SOIREE SPECIAL...,135,17,27,3,0,0.951807
104,ham,Umma my life and vava umma love you lot dear,44,10,0,0,0,0.028571
3564,ham,WHORE YOU ARE UNBELIEVABLE.,27,4,0,0,0,1.000000
1845,ham,Becoz its <#> jan whn al the post ofice is i...,102,22,0,0,0,0.013699


In [15]:
vectorizer = CountVectorizer(stop_words="english", max_features=500)
X_train_words = pd.DataFrame(
    vectorizer.fit_transform(X_train_engraw["message"]).toarray(),
    columns=vectorizer.get_feature_names_out(), index=X_train_engraw.index
)
X_test_words = pd.DataFrame(
    vectorizer.transform(X_test_engraw["message"]).toarray(),
    columns=vectorizer.get_feature_names_out(), index=X_test_engraw.index
)

In [16]:
X_train_words

,000,03,08000930705,10,100,1000,10p,150p,150ppm,16,...,xxx,ya,yar,yeah,year,years,yes,yesterday,yo,yup
331,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
383,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
573,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1070,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
598,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
104,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3564,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1845,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
# combining word counts and engineered features into one table
X_train = pd.concat([X_train_words, X_train_engraw[engineered_cols]], axis=1)
X_test = pd.concat([X_test_words, X_test_engraw[engineered_cols]], axis=1)
print("Combined table:", X_train.shape)

Combined table: (4135, 506)


In [18]:
X_train

,000,03,08000930705,10,100,1000,10p,150p,150ppm,16,...,yes,yesterday,yo,yup,char_len,word_len,digit_count,exclaim_count,currency_count,capital_ratio
331,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,45,10,0,1,0,0.093750
383,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,47,9,0,0,0,0.055556
573,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,46,10,0,0,0,0.027027
1346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,27,7,1,0,0,0.052632
1070,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,71,17,0,0,0,0.060000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
598,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,135,17,27,3,0,0.951807
104,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,44,10,0,0,0,0.028571
3564,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,27,4,0,0,0,1.000000
1845,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,102,22,0,0,0,0.013699


In [19]:
# chi2 feature selection
selector = SelectKBest(chi2, k=100)
X_train_selected = pd.DataFrame(selector.fit_transform(X_train, y_train),
                                 columns=X_train.columns[selector.get_support()])
X_test_selected = pd.DataFrame(selector.transform(X_test),
                                columns=X_test.columns[selector.get_support()])
print("After chi2 selection:", X_train_selected.shape)

After chi2 selection: (4135, 100)


In [20]:
X_train_selected

,000,03,08000930705,10,100,1000,10p,150p,150ppm,16,...,weekly,win,wk,won,www,char_len,word_len,digit_count,exclaim_count,currency_count
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,45.0,10.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,47.0,9.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,46.0,10.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,27.0,7.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,71.0,17.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4130,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,135.0,17.0,27.0,3.0,0.0
4131,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,44.0,10.0,0.0,0.0,0.0
4132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,27.0,4.0,0.0,0.0,0.0
4133,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,102.0,22.0,0.0,0.0,0.0


In [21]:
X_test_selected

,000,03,08000930705,10,100,1000,10p,150p,150ppm,16,...,weekly,win,wk,won,www,char_len,word_len,digit_count,exclaim_count,currency_count
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,159.0,33.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,38.0,8.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,39.0,9.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,48.0,8.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,50.0,12.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,156.0,27.0,22.0,1.0,0.0
1030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23.0,5.0,0.0,0.0,0.0
1031,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,148.0,23.0,19.0,0.0,2.0
1032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,123.0,22.0,1.0,0.0,0.0


In [22]:
# saving
joblib.dump(vectorizer, "models/vectorizer.joblib")
joblib.dump(selector, "models/selector.joblib")
X_train_selected.to_csv("data/processed/X_train.csv", index=False)
X_test_selected.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

In [ ]:
mlflow.set_experiment("spam_filter_benchmarks_engineered")
mlflow.sklearn.autolog()

results_autolog = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        test_metrics = {
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1": f1_score(y_test, predictions, zero_division=0),
        }
        mlflow.log_metrics({f"test_{k.lower()}": v for k, v in test_metrics.items()})
        results_autolog.append({"Model": name, **test_metrics})

mlflow.sklearn.autolog(disable=True)

In [24]:
results_df = pd.DataFrame(results_autolog).round(3)
print(results_df.to_string(index=False))

                  Model  Accuracy  Precision  Recall    F1
Majority-class baseline     0.873      0.000   0.000 0.000
          Decision Tree     0.979      0.943   0.885 0.913
                    KNN     0.981      0.966   0.878 0.920
            Naive Bayes     0.984      0.952   0.916 0.934
Random Forest (bagging)     0.987      0.992   0.908 0.948
    AdaBoost (boosting)     0.984      0.960   0.908 0.933
    Logistic Regression     0.988      0.992   0.916 0.952
